# Assignment 10

In [106]:
import numpy as np
import cv2
import mediapipe as mp
import pandas as pd
import os
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [ ]:

# def euclidean(x1, y1, z1, x2, y2, z2):
#     return np.sqrt((x1-x2)**2 + (y1-y2)**2 + (z1-z2)**2)

# def euclidean(x1, y1, x2, y2):
#     return np.sqrt((x1-x2)**2 + (y1-y2)**2)

def euclidean(x1, x2):
    return np.abs((x1-x2))

video_tags = {"A":159, "B":22}
base_dir_mediapipe = "../../MainProject/Assignment10/data/csv_of_all_videos/"
base_dir_kinect = "../../MainProject/Assignment9/data/kinect_good_preprocessed/"

for letter, number in video_tags.items():
    for i in range(1, number + 1):
        mediapipe_dir = base_dir_mediapipe + f"{letter}{i}_mediapipe.csv"
        kinect_dir = base_dir_kinect + f"{letter}{i}_kinect.csv"

        df_mediapipe = pd.read_csv(mediapipe_dir)
        df_kinect = pd.read_csv(kinect_dir)

        # Kinect files have a column named " head_x" instead of "head_x" for some reason...
        df_kinect = df_kinect.rename(columns={" head_x":"head_x"})

        # Only keep frames that exist in both mediapipe and kinect DataFrames
        frames = df_kinect["FrameNo"].values
        df_mediapipe = df_mediapipe.loc[df_mediapipe.index.isin(frames)]
        df_mediapipe = df_mediapipe.reset_index(drop=True)

        columns = df_kinect.columns
        x_columns = [col for col in columns if "_x" in col]
        y_columns = [col for col in columns if "_y" in col]
        z_columns = [col for col in columns if "_z" in col]

        shoulder_length_kinect = euclidean(df_kinect["left_shoulder_x"],
                                           df_kinect["right_shoulder_x"])
        shoulder_length_mediapipe = euclidean(df_mediapipe["left_shoulder_x"],
                                              df_mediapipe["right_shoulder_x"])
        x_scaling_factor = shoulder_length_kinect / shoulder_length_mediapipe

        torso_length_kinect = euclidean(df_kinect["left_shoulder_x"],
                                        df_kinect["left_hip_x"])
        torso_length_mediapipe = euclidean(df_mediapipe["left_shoulder_x"],
                                        df_mediapipe["left_hip_x"])
        y_scaling_factor = torso_length_kinect / torso_length_mediapipe

        df_kinect = df_kinect.drop("FrameNo", axis=1)
        df_mediapipe = df_mediapipe.drop("FrameNo", axis=1)

        for x in x_columns:
            df_mediapipe[x] *= x_scaling_factor

        for y in y_columns:
            df_mediapipe[y] *= y_scaling_factor

        df_difference = np.abs(df_kinect - df_mediapipe)
        print(x_scaling_factor)
        print(y_scaling_factor)
        break
    break

print(f"{'Node':<25} {'Kinect (m)':>10} {'MediaPipe (m)':>15} {'Error (m)':>12} {'Error (%)':>10}")
print("-" * 75)
for column in df_difference.columns:
    print(f"{column:<25} {df_kinect[column][0]:>10.2f} {df_mediapipe[column][0]:>15.2f} {df_difference[column][0]:>12.2f}")


Node                      Kinect (m)   MediaPipe (m)    Error (m)  Error (%)
---------------------------------------------------------------------------
head_x                          0.01            0.07         0.05
head_y                          0.77           -0.99         1.77
head_z                          0.04           -0.27         0.30
left_shoulder_x                -0.14            0.23         0.37
left_shoulder_y                 0.56           -0.72         1.28
left_shoulder_z                 0.04           -0.10         0.14
left_elbow_x                   -0.24            0.36         0.59
left_elbow_y                    0.79           -1.06         1.85
left_elbow_z                    0.02           -0.17         0.18
right_shoulder_x                0.16           -0.07         0.24
right_shoulder_y                0.53           -0.72         1.25
right_shoulder_z                0.03           -0.09         0.12
right_elbow_x                   0.28           -0.18   